# 생활시설 배치 결과 자동 집계

지정한 생활시설 배치 비율과 읍면동 목록을 기준으로 `selected_multi_facility_plan.csv`를 자동으로 찾습니다.

다음 두 종류의 표를 출력합니다.

1. 선택한 읍면동 전체의 생활지수별 시설 배치 수
2. 선택한 읍면동 각각의 생활지수별 시설유형 배치 수

시설유형이 배치되지 않은 경우도 0으로 표시합니다.

## 사용 방법

아래 설정 cell에서 다음 세 항목만 수정하면 됩니다.

- `PLAN_SOURCE`: 특정 CSV 파일 또는 `multi_facility_unmet_population` 폴더 경로
- `FACILITY_SHARE_PERCENT`: 시설 배치 비율. 예: 100
- `TARGET_ADMS`: 분석할 읍면동 리스트

`PLAN_SOURCE`에 폴더를 지정하면 해당 폴더 아래에서 `{배치비율}pct/selected_multi_facility_plan.csv`를 자동으로 찾습니다. 여러 파일이 발견되면 잘못된 파일을 임의로 선택하지 않고 경로를 출력합니다.

In [1]:
from pathlib import Path
import warnings

import pandas as pd
from IPython.display import display, Markdown

# ==============================
# 사용자 설정
# ==============================
IS_COLAB = False

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    IS_COLAB = True

except ModuleNotFoundError:
    pass


# 환경별 기본 경로 설정
if IS_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/Cheonan")
else:
    # 로컬에서는 현재 노트북이 실행되는 폴더를 기본 경로로 사용
    BASE_DIR = Path.cwd()


# BASE_DIR을 기준으로 한 공통 상대경로
RELATIVE_PLAN_SOURCE = Path(
    "clustering_outputs"
    / Path("lscp_siting_outputs")
    / Path("multi_facility_unmet_population")
    / "1000m_base"
)

PLAN_SOURCE = BASE_DIR / RELATIVE_PLAN_SOURCE


if not PLAN_SOURCE.exists():
    raise FileNotFoundError(
        f"배치계획 폴더를 찾을 수 없습니다.\n"
        f"확인한 경로: {PLAN_SOURCE}"
    )

print(f"실행 환경: {'Colab' if IS_COLAB else '로컬'}")
print(f"기본 경로: {BASE_DIR}")
print(f"배치계획 경로: {PLAN_SOURCE}")

# 시설 배치 비율: 10, 20, ..., 100 중 하나를 입력합니다.
FACILITY_SHARE_PERCENT = 40

# 분석할 읍면동을 입력합니다. 빈 리스트이면 CSV에 포함된 모든 읍면동을 분석합니다.
# 기존 클러스터 3: '성환읍', '직산읍', '성거읍', '목천읍', '풍세면'
# 기존 클러스터 4: '입장면', '북면', '병천면', '동면', '성남면', '수신면', '광덕면'
TARGET_ADMS = [
    '성남면', '광덕면',
]

# 표를 CSV로도 저장하려면 True로 바꿉니다.
SAVE_SUMMARY_CSV = True

if not isinstance(FACILITY_SHARE_PERCENT, int) or not 0 < FACILITY_SHARE_PERCENT <= 100:
    raise ValueError('FACILITY_SHARE_PERCENT는 1 이상 100 이하의 정수여야 합니다.')


실행 환경: 로컬
기본 경로: C:\Users\심현석\Documents\test\Cheonan
배치계획 경로: C:\Users\심현석\Documents\test\Cheonan\clustering_outputs\lscp_siting_outputs\multi_facility_unmet_population\1000m_base


In [2]:
INDEX_ORDER = [
    'life_welfare_index',
    'commerce_index',
    'medical_index',
    'education_index',
    'leisure_index',
]

INDEX_LABELS = {
    'life_welfare_index': '생활복지',
    'commerce_index': '상업',
    'medical_index': '의료',
    'education_index': '교육',
    'leisure_index': '여가',
}

FACILITY_TYPES_BY_INDEX = {
    'life_welfare_index': {
        'agency': '행정기관',
        'park': '공원',
        'welfare_facilities_for_elderly': '노인복지시설',
    },
    'commerce_index': {
        'convstore': '편의점',
        'market': '전통시장',
        'shoppingmall': '쇼핑몰',
    },
    'medical_index': {
        'hospital': '병원',
        'pharmacy': '약국',
    },
    'education_index': {
        'school': '학교',
    },
    'leisure_index': {
        'library': '도서관',
        'museum': '박물관·미술관·기념관',
        'sportsandresort': '체육시설·공공리조트',
        'theater': '극장',
    },
}

REQUIRED_COLUMNS = {'target_adm', 'GRID_CD', 'index_name', 'facility_label'}


def find_plan_csv(source: Path, share_percent: int) -> Path:
    """파일 또는 폴더 경로에서 지정된 배치 비율의 계획 CSV를 찾습니다."""
    source = source.expanduser()
    expected_name = 'selected_multi_facility_plan.csv'
    expected_folder = f'{share_percent}pct'

    if source.is_file():
        if source.name != expected_name:
            warnings.warn(
                f'지정한 파일명이 {expected_name}이 아니지만, 해당 파일을 사용합니다: {source.name}'
            )
        return source

    if not source.exists():
        raise FileNotFoundError(f'지정한 경로가 존재하지 않습니다: {source}')
    if not source.is_dir():
        raise NotADirectoryError(f'파일 또는 폴더가 아닙니다: {source}')

    direct = source / expected_folder / expected_name
    if direct.exists():
        return direct

    matches = sorted(source.rglob(f'{expected_folder}/{expected_name}'))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f'{share_percent}pct/{expected_name}을 찾지 못했습니다. 검색 시작 경로: {source}'
        )
    attempted = '\n'.join(f'- {path}' for path in matches)
    raise RuntimeError(
        '동일한 배치 비율의 CSV가 여러 개 발견되었습니다. '
        '3x3_base 또는 1000m_base 폴더까지 포함한 경로를 지정하세요.\n' + attempted
    )


PLAN_CSV = find_plan_csv(PLAN_SOURCE, FACILITY_SHARE_PERCENT)
plan = pd.read_csv(PLAN_CSV, encoding='utf-8-sig')
missing = sorted(REQUIRED_COLUMNS - set(plan.columns))
if missing:
    raise ValueError(f'계획 CSV에 필요한 컬럼이 없습니다: {missing}')

plan['target_adm'] = plan['target_adm'].astype(str)
plan['index_name'] = plan['index_name'].astype(str)
plan['facility_label'] = plan['facility_label'].astype(str)

available_adms = plan['target_adm'].dropna().unique().tolist()
if TARGET_ADMS:
    missing_adms = [adm for adm in TARGET_ADMS if adm not in available_adms]
    if missing_adms:
        warnings.warn(f'CSV에서 찾지 못한 읍면동은 제외합니다: {missing_adms}')
    selected_adms = [adm for adm in TARGET_ADMS if adm in available_adms]
else:
    selected_adms = sorted(available_adms)

if not selected_adms:
    raise ValueError('분석할 읍면동이 없습니다.')

selected_plan = plan.loc[plan['target_adm'].isin(selected_adms)].copy()
if selected_plan.empty:
    raise ValueError('선택한 읍면동에 해당하는 배치 기록이 없습니다.')

print(f'사용 CSV: {PLAN_CSV}')
print(f'시설 배치 비율: {FACILITY_SHARE_PERCENT}%')
print(f"분석 읍면동: {', '.join(selected_adms)}")
print(f'분석 배치 기록 수: {len(selected_plan):,}개')


사용 CSV: C:\Users\심현석\Documents\test\Cheonan\clustering_outputs\lscp_siting_outputs\multi_facility_unmet_population\1000m_base\40pct\selected_multi_facility_plan.csv
시설 배치 비율: 40%
분석 읍면동: 성남면, 광덕면
분석 배치 기록 수: 65개


## 1. 읍면동 전체의 생활지수별 시설 배치 수

선택한 읍면동들의 배치 기록을 합산합니다. 각 행은 하나의 시설 배치 기록입니다.

In [3]:
index_count_table = (
    selected_plan.groupby(['target_adm', 'index_name'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=selected_adms, columns=INDEX_ORDER, fill_value=0)
)
index_count_table.columns = [INDEX_LABELS[column] for column in index_count_table.columns]
index_count_table['합계'] = index_count_table.sum(axis=1)
index_count_table.loc['전체'] = index_count_table.sum(axis=0)
index_count_table.index.name = '읍면동'

display(Markdown('### 읍면동별 생활지수별 시설 배치 수'))
display(index_count_table.style.format('{:,.0f}'))


### 읍면동별 생활지수별 시설 배치 수

,생활복지,상업,의료,교육,여가,합계
읍면동,,,,,,
성남면,4,4,4,4,5,21
광덕면,9,8,9,9,9,44
전체,13,12,13,13,14,65


## 2. 읍면동별 생활지수·시설유형 배치 수

각 읍면동에 대해 생활지수별 관련 시설유형을 모두 표시합니다. 배치되지 않은 시설유형도 0으로 표시합니다.

In [4]:
facility_count_tables = {}
summary_output_dir = PLAN_CSV.parent / 'facility_plan_analysis_tables'
if SAVE_SUMMARY_CSV:
    summary_output_dir.mkdir(parents=True, exist_ok=True)

for adm_name in selected_adms:
    adm_plan = selected_plan.loc[selected_plan['target_adm'].eq(adm_name)]
    rows = []
    for index_name in INDEX_ORDER:
        index_plan = adm_plan.loc[adm_plan['index_name'].eq(index_name)]
        counts = index_plan['facility_label'].value_counts()
        for facility_code, facility_label in FACILITY_TYPES_BY_INDEX[index_name].items():
            rows.append({
                '생활지수': INDEX_LABELS[index_name],
                '시설유형': facility_label,
                '배치수': int(counts.get(facility_code, 0)),
            })
    table = pd.DataFrame(rows)
    facility_count_tables[adm_name] = table
    display(Markdown(f'### {adm_name}: 생활지수별 시설유형 배치 수'))
    display(table.style.format({'배치수': '{:,.0f}'}))

if SAVE_SUMMARY_CSV:
    index_count_table.reset_index().to_csv(
        summary_output_dir / 'selected_adms_index_facility_counts.csv',
        index=False, encoding='utf-8-sig'
    )
    detail_table = pd.concat(
        [table.assign(읍면동=adm_name) for adm_name, table in facility_count_tables.items()],
        ignore_index=True
    )
    detail_table.to_csv(
        summary_output_dir / 'selected_adms_facility_type_counts.csv',
        index=False, encoding='utf-8-sig'
    )
    print(f'요약표 저장 폴더: {summary_output_dir}')


### 성남면: 생활지수별 시설유형 배치 수

,생활지수,시설유형,배치수
0,생활복지,행정기관,4
1,생활복지,공원,0
2,생활복지,노인복지시설,0
3,상업,편의점,0
4,상업,전통시장,4
5,상업,쇼핑몰,0
6,의료,병원,0
7,의료,약국,4
8,교육,학교,4
9,여가,도서관,0


### 광덕면: 생활지수별 시설유형 배치 수

,생활지수,시설유형,배치수
0,생활복지,행정기관,9
1,생활복지,공원,0
2,생활복지,노인복지시설,0
3,상업,편의점,0
4,상업,전통시장,8
5,상업,쇼핑몰,0
6,의료,병원,0
7,의료,약국,9
8,교육,학교,9
9,여가,도서관,0


요약표 저장 폴더: C:\Users\심현석\Documents\test\Cheonan\clustering_outputs\lscp_siting_outputs\multi_facility_unmet_population\1000m_base\40pct\facility_plan_analysis_tables
